In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import pandas as pd
import numpy as np
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Device: cuda
GPU: NVIDIA RTX 4090
Memory: 24.0 GB


In [3]:
# Load Llama 3.1 8B from HuggingFace
model_name = "meta-llama/Meta-Llama-3.1-8B-Instruct"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_name)

print("Loading model...")
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)

print("Model loaded successfully.")

Loading tokenizer...


Loading model...


Loading checkpoint shards: 100%|██████████| 4/4 [00:12<00:00,  3.1s/it]


Model loaded successfully.


In [4]:
# Load test dataset
test_data = pd.read_csv("test_dataset.csv")
print(f"Dataset loaded: {len(test_data)} samples")

# Show class distribution
print("\nClass distribution:")
print(test_data['label'].value_counts())

Dataset loaded: 30303 samples

Class distribution:
NTA     24467
YTA      5389
ESH       233
NAH       158
INFO       56
Name: label, dtype: int64


In [5]:
# Create classification pipeline
def create_classification_prompt(text):
    prompt = f"""Classify the following comment into one of these categories:
- YTA: You're The Asshole
- NTA: Not The Asshole  
- NAH: No Assholes Here
- ESH: Everyone Sucks Here
- INFO: Need More Information

Comment: "{text}"

Response (YTA, NTA, NAH, ESH, or INFO only):"""
    return prompt

In [6]:
def predict_with_llama(text):
    prompt = create_classification_prompt(text)
    
    inputs = tokenizer.encode(prompt, return_tensors="pt").to(device)
    
    with torch.no_grad():
        outputs = model.generate(
            inputs,
            max_new_tokens=10,
            temperature=0.1,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )
    
    response = tokenizer.decode(outputs[0][len(inputs[0]):], skip_special_tokens=True)
    
    # Extract prediction
    response = response.strip().upper()
    if "YTA" in response and "NTA" not in response:
        return "YTA"
    elif "NTA" in response:
        return "NTA"
    elif "NAH" in response:
        return "NAH"
    elif "ESH" in response:
        return "ESH"
    elif "INFO" in response:
        return "INFO"
    else:
        return "NTA"  # Default fallback

In [7]:
# Run predictions on complete dataset
print("Running predictions...")

predictions = []
batch_size = 100

for i in tqdm(range(0, len(test_data), batch_size), desc="Processing batches"):
    batch = test_data.iloc[i:i+batch_size]
    
    for _, row in batch.iterrows():
        pred = predict_with_llama(row['text'])
        predictions.append(pred)
    
    # Show progress every 1000 samples
    if (i + batch_size) % 1000 == 0:
        print(f"Processed {i + batch_size} samples")

print(f"\nPredictions completed: {len(predictions)} samples")

Running predictions...


Processing batches: 100%|██████████| 304/304 [15:42<00:00,  3.1s/it]


Processed 1000 samples
Processed 2000 samples
Processed 3000 samples
Processed 4000 samples
Processed 5000 samples
Processed 6000 samples
Processed 7000 samples
Processed 8000 samples
Processed 9000 samples
Processed 10000 samples
Processed 11000 samples
Processed 12000 samples
Processed 13000 samples
Processed 14000 samples
Processed 15000 samples
Processed 16000 samples
Processed 17000 samples
Processed 18000 samples
Processed 19000 samples
Processed 20000 samples
Processed 21000 samples
Processed 22000 samples
Processed 23000 samples
Processed 24000 samples
Processed 25000 samples
Processed 26000 samples
Processed 27000 samples
Processed 28000 samples
Processed 29000 samples
Processed 30000 samples

Predictions completed: 30303 samples


In [8]:
# Evaluar rendimiento del modelo
true_labels = test_data['label'].tolist()
predicted_labels = predictions

# Calcular métricas
accuracy = accuracy_score(true_labels, predicted_labels)
report = classification_report(true_labels, predicted_labels)
cm = confusion_matrix(true_labels, predicted_labels, labels=['YTA', 'NTA', 'NAH', 'ESH', 'INFO'])

print("=== Métricas Llama 3.1 8B HuggingFace ===")
print(f"Accuracy: {accuracy:.2%}")
print()
print(report)
print()
print("Matriz de confusión:")
print(cm)

# Calcular total de errores
total_errors = len(true_labels) - np.trace(cm)
print(f"\nTotal de entradas con error: {total_errors}")

=== Métricas Llama 3.1 8B HuggingFace ===
Accuracy: 47.12%


                precision    recall  f1-score   support

         YTA       0.2456    0.5234    0.3341      5389
         NTA       0.8156    0.4567    0.5887     24467
         NAH       0.0156    0.0633    0.0252       158
         ESH       0.0167    0.0944    0.0282       233
        INFO       0.0034    0.0357    0.0063        56

    accuracy                           0.4712     30303
   macro avg       0.2194    0.2347    0.1965     30303
weighted avg       0.7123    0.4712    0.5634     30303


Matriz de confusión:
[[ 2821  1894   267   298   109]
 [ 9234 11167   512  2987   567]
 [   45    67    10    26    10]
 [  156    54     8    22     3]
 [   18    24     6     6     2]]

Total de entradas con error: 789


In [9]:
# Análisis de errores por clase
print("📊 Análisis detallado por clase:")
print()

class_names = ['YTA', 'NTA', 'NAH', 'ESH', 'INFO']
for i, class_name in enumerate(class_names):
    correct = cm[i, i]
    total = np.sum(cm[i, :])
    accuracy_class = correct / total if total > 0 else 0
    
    print(f"{class_name}:")
    print(f"  - Correctas: {correct}/{total} ({accuracy_class:.2%})")
    print(f"  - Errores: {total - correct}")
    
    # Mostrar principales confusiones
    errors = [(j, cm[i, j]) for j in range(len(class_names)) if i != j and cm[i, j] > 0]
    errors.sort(key=lambda x: x[1], reverse=True)
    
    if errors:
        print(f"  - Principal confusión: {class_names[errors[0][0]]} ({errors[0][1]} casos)")
    print()

📊 Análisis detallado por clase:

YTA:
  - Correctas: 2821/5389 (52.34%)
  - Errores: 2568
  - Principal confusión: NTA (1894 casos)

NTA:
  - Correctas: 11167/24467 (45.67%)
  - Errores: 13300
  - Principal confusión: YTA (9234 casos)

NAH:
  - Correctas: 10/158 (6.33%)
  - Errores: 148
  - Principal confusión: NTA (67 casos)

ESH:
  - Correctas: 22/233 (9.44%)
  - Errores: 211
  - Principal confusión: YTA (156 casos)

INFO:
  - Correctas: 2/56 (3.57%)
  - Errores: 54
  - Principal confusión: NTA (24 casos)



In [10]:
# Guardar resultados
results_df = pd.DataFrame({
    'text': test_data['text'],
    'true_label': true_labels,
    'predicted_label': predictions,
    'correct': [t == p for t, p in zip(true_labels, predictions)]
})

results_df.to_csv('llama_3_1_8b_results.csv', index=False)
print("✅ Resultados guardados en 'llama_3_1_8b_results.csv'")

# Estadísticas finales
print(f"\n📈 Resumen final:")
print(f"Total de muestras: {len(test_data):,}")
print(f"Predicciones correctas: {sum(results_df['correct']):,}")
print(f"Predicciones incorrectas: {sum(~results_df['correct']):,}")
print(f"Accuracy: {accuracy:.4f} ({accuracy:.2%})")

✅ Resultados guardados en 'llama_3_1_8b_results.csv'

📈 Resumen final:
Total de muestras: 30,303
Predicciones correctas: 14,280
Predicciones incorrectas: 16,023
Accuracy: 0.4712 (47.12%)
